# Neutrino GPU proving gate on Colab

Runs the real CUDA `EvidenceProof → block → chunk → History` acceptance gate and the guest instruction profile on a Colab GPU runtime.

**Before you start**
- Runtime → Change runtime type → GPU. Prefer a **high-RAM** runtime (A100 or L4). A free-tier T4 with ~12 GiB RAM is very likely to be OOM-killed during compressed proving.
- SP1 stage caches and logs live on Google Drive so an interrupted gate resumes by re-running the cells. The Rust build stays on local disk because Drive is mounted without execute permission.
- The repository must be reachable from Colab. Set `REPO_URL` and `COMMIT` below. For a private repo, use a token URL or upload the tree to Drive and set `REPO_URL` to that path.


In [ ]:
#@title Parameters
REPO_URL = "https://github.com/Rorical/Neutrino"  #@param {type:"string"}
COMMIT = "main"  #@param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/neutrino-gpu"  #@param {type:"string"}
SHARD_SIZE = 1048576  #@param {type:"integer"}
RUN_CPU_GATE = False  #@param {type:"boolean"}
import os
os.environ.update({"REPO_URL": REPO_URL, "COMMIT": COMMIT, "OUT_DIR": OUT_DIR,
                   "SHARD_SIZE": str(SHARD_SIZE), "SKIP_CPU": "0" if RUN_CPU_GATE else "1"})


In [ ]:
#@title Mount Drive and check the GPU
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(os.environ["OUT_DIR"], exist_ok=True)
!nvidia-smi
!free -g | head -2
!nproc


In [ ]:
#@title Fetch the repository (idempotent, always the latest for a branch)
%%bash
set -euo pipefail
if [ -d /content/neutrino/.git ]; then
  cd /content/neutrino && git fetch --all --quiet --prune
else
  git clone --quiet "$REPO_URL" /content/neutrino && cd /content/neutrino
fi
cd /content/neutrino
# A branch name resolves to its remote tip, not to a stale local branch.
if git rev-parse --verify --quiet "origin/$COMMIT" >/dev/null; then
  git checkout --quiet --detach "origin/$COMMIT"
else
  git checkout --quiet --detach "$COMMIT"
fi
git log --oneline -1


In [ ]:
#@title Install toolchains, build with CUDA, run the gate (resumable)
%%bash
set -euo pipefail
cd /content/neutrino
# Build on local disk (Drive is mounted noexec); SP1 stage caches and logs go to OUT_DIR.
exec bash scripts/colab-gpu-gate.sh 2>&1 | tee -a "$OUT_DIR/run.log"


In [ ]:
#@title Summarise results
%%bash
set -u
cd "$OUT_DIR"
echo "== wall time and peak memory =="
grep -E 'cuda gate wall seconds|Maximum resident set size|Elapsed \(wall clock\)' cuda-evidence-gate.log 2>/dev/null || echo "gate log not found"
echo
echo "== SP1 stage timings (last 40 lines mentioning prove/compress/verify) =="
grep -iE 'prov|compress|verify|shard|recurs' cuda-evidence-gate.log 2>/dev/null | tail -40
echo
echo "== guest instruction profile =="
cat profile.log 2>/dev/null || echo "profile log not found"
echo
echo "== test verdicts =="
grep -E '^test result' cuda-evidence-gate.log cpu-consensus-gate.log 2>/dev/null


## If it fails

- **Process killed, no panic in the log**: out of memory. Lower `SHARD_SIZE` to `524288` in the parameters cell and re-run the gate cell; stage caches are kept.
- **`Failed to download CUDA server`**: SP1 downloads `sp1-gpu-server` from GitHub releases; retry, or check the runtime has outbound network access.
- **Session disconnected**: re-run the cells in order. SP1 stage caches on Drive let the gate resume where it stopped; the Rust build itself is redone on local disk because Drive cannot execute binaries.

Send `cuda-evidence-gate.log` and `profile.log` back for analysis.
